# Suivre les données et le fonctionnement de l'API de scoring

Ce notebook est un **PoC de monitoring** : il aide à repérer des changements dans les clients reçus,
dans les décisions du modèle et dans le fonctionnement du service.

Il répond à trois questions :

| Question | Données actuelles | Référence de comparaison |
|---|---|---|
| Les profils clients changent-ils ? | Entrées des prédictions réussies récentes | CSV train préparé, avant imputation |
| Le modèle donne-t-il des scores ou des décisions différents ? | Scores et classes enregistrés par l'API | Prédictions du même pipeline sur le train |
| Le service se dégrade-t-il ? | Les 100 derniers appels /predict, tous statuts | Les 100 appels précédents |

**Dérive des données** signifie changement de distribution, par exemple davantage de petits crédits.
Cela ne prouve pas que le modèle prédit moins bien : il faudrait connaître les défauts réels pour mesurer sa performance.

Le parcours est : **charger → sélectionner les appels → calculer la référence des scores → préparer les lots →
produire un rapport Evidently → lire les décisions et les indicateurs opérationnels**.

Exécuter de haut en bas, une cellule à la fois. Le notebook lit le CSV et SQLite sans les modifier.
Il ne génère aucun appel API et ne se met pas à jour tout seul : il faut relancer la lecture pour analyser de nouveaux appels.

**Préparation :** installer les dépendances depuis la racine du projet, puis choisir le noyau de sa .venv :

```powershell
uv pip install --python .venv\Scripts\python.exe -r requirements.txt
```

Le notebook ne lance aucune installation.

## 1. Choisir ce que nous allons surveiller

**La cellule suivante** importe les outils, retrouve la racine du projet et centralise les paramètres.

| Paramètre | Ce qu'il permet de changer |
|---|---|
| CHEMIN_TRAIN / CHEMIN_SQLITE | Les sources à lire |
| VERSION_MODELE | La version des réussites retenues pour le suivi des données et des scores |
| NOMBRE_APPELS | La taille de chaque fenêtre glissante : 100 par défaut |
| VARIABLES | Les entrées numériques surveillées |
| MIN_OBSERVATIONS | Le minimum de lignes ou de valeurs renseignées pour une analyse : 50 |
| TAILLE_LOT_PREDICTION | Le nombre de lignes traitées à la fois pour calculer les scores du train |
| SEUIL_WASSERSTEIN / SEUIL_P | Les seuils des deux méthodes de dérive, expliqués à l'étape 6 |

Nous commençons avec trois variables faciles à expliquer :
**AMT_CREDIT**, montant du crédit ; **AMT_INCOME_TOTAL**, revenu ; **EXT_SOURCE_2**, score externe.
Ce choix pédagogique ne couvre pas toutes les variables importantes du modèle.

DO_NOT_TRACK désactive la télémétrie Evidently avant son import.
Si vous changez un paramètre, relancez les cellules qui en dépendent.

In [ ]:
import os
os.environ["DO_NOT_TRACK"] = "1"  # Désactive la télémétrie Evidently.

import json
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display
from evidently import DataDefinition, Dataset, Report
from evidently.metrics import MissingValueCount, ValueDrift

# Le notebook peut être ouvert depuis la racine ou depuis notebooks/.
RACINE = Path.cwd()
if not (RACINE / "app/main.py").is_file():
    RACINE = RACINE.parent
if not (RACINE / "app/main.py").is_file():
    raise FileNotFoundError("Ouvrir le notebook depuis la racine du projet ou notebooks/.")

CHEMIN_TRAIN = RACINE / "data/processed/application_train_final_encoded.csv"
CHEMIN_SQLITE = RACINE / "data/api_calls.sqlite3"
VERSION_MODELE = "4"
NOMBRE_APPELS = 100
TAILLE_LOT_PREDICTION = 10000
VARIABLES = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "EXT_SOURCE_2"]
MIN_OBSERVATIONS = 50
SEUIL_WASSERSTEIN = 0.1
SEUIL_P = 0.05 / (len(VARIABLES) + 1)

## 2. Lire les valeurs de référence du train

**Pourquoi ?** Nous avons choisi le train préparé comme portrait initial de la population.
Ses valeurs manquantes doivent rester visibles : nous ne les remplaçons pas pour comparer les entrées.

**La cellule suivante** charge uniquement les trois colonnes surveillées, les convertit en nombres
et affiche le nombre de clients ainsi qu'un tableau descriptif.

**À lire :** count est le nombre de valeurs renseignées, mean la moyenne, std la dispersion.
La colonne 50% est la médiane : la moitié des valeurs sont inférieures ou égales.
Si count est inférieur au nombre de clients, la variable contient des valeurs manquantes.
TARGET et l'identifiant client ne sont pas nécessaires ici.

In [2]:
if not CHEMIN_TRAIN.is_file():
    raise FileNotFoundError(f"Fichier d’entraînement absent : {CHEMIN_TRAIN}")

reference = pd.read_csv(CHEMIN_TRAIN, usecols=VARIABLES)[VARIABLES]
reference = reference.astype("float64")
print("Nombre de lignes de référence :", len(reference))
display(reference.describe().T)

Nombre de lignes de référence : 307510


,count,mean,std,min,25%,50%,75%,max
AMT_CREDIT,307510.0,599026.118515,402491.426040,4.500000e+04,270000.000000,513531.000000,808650.000000,4.050000e+06
AMT_INCOME_TOTAL,307510.0,168417.992784,108811.807346,2.565000e+04,112500.000000,147150.000000,202500.000000,1.800009e+07
EXT_SOURCE_2,306850.0,0.514394,0.191059,8.173617e-08,0.392457,0.565964,0.663617,8.549997e-01


## 3. Lire deux fenêtres d'appels dans SQLite

**La cellule suivante** ouvre SQLite en lecture seule (mode=ro), puis ferme la connexion après lecture.
Elle prend jusqu'à 200 appels /predict, triés du plus récent au plus ancien :

- fenetre : les 100 derniers appels, utilisés pour le suivi actuel ;
- fenetre_precedente : les 100 juste avant, réservés à la comparaison opérationnelle de l'étape 8.

Les deux fenêtres ne se chevauchent pas. Elles sont sélectionnées **avant tout filtrage des statuts**.
Les dates affichées décrivent les appels retenus ; aucune période calendaire n'est imposée.

Pour préparer les entrées destinées à Evidently, nous gardons ensuite les réussites de VERSION_MODELE.
Les rejets et les pannes n'ont pas de données client enregistrées à analyser.
La colonne input_json contient les valeurs avant imputation ; ses null deviennent des NaN pandas.

**À lire :** volume actuel, réussites, rejets, pannes et réussites de la bonne version.
Le volume final du rapport peut encore diminuer à l'étape 5, si certains seuils diffèrent.
Avec moins de 100 appels disponibles, la fenêtre est incomplète et le notebook le signale.

In [3]:
if not CHEMIN_SQLITE.is_file():
    raise FileNotFoundError(f"Base SQLite absente : {CHEMIN_SQLITE}")
if not isinstance(NOMBRE_APPELS, int) or isinstance(NOMBRE_APPELS, bool) or NOMBRE_APPELS <= 0:
    raise ValueError("NOMBRE_APPELS doit être un entier strictement positif.")

with closing(sqlite3.connect(CHEMIN_SQLITE.resolve().as_uri() + "?mode=ro", uri=True)) as connexion:
    historique_recent = pd.read_sql_query(
        """SELECT timestamp_utc, status, model_version, input_json, score, prediction, threshold, duration_ms FROM api_calls
           WHERE operation = 'predict'
           ORDER BY timestamp_utc DESC, rowid DESC
           LIMIT ?""",
        connexion,
        params=(2 * NOMBRE_APPELS,),
    )

fenetre = historique_recent.iloc[:NOMBRE_APPELS].copy()
fenetre_precedente = historique_recent.iloc[NOMBRE_APPELS:].copy()

print("Appels /predict dans la fenêtre :", len(fenetre), "/", NOMBRE_APPELS)
for statut, libelle in [("success", "Réussites"), ("rejected", "Rejets"), ("error", "Pannes")]:
    print(libelle, ":", int(fenetre["status"].eq(statut).sum()))
if len(fenetre) < NOMBRE_APPELS:
    print("Fenêtre incomplète : tous les appels disponibles sont retenus.")
if not fenetre.empty:
    dates = pd.to_datetime(fenetre["timestamp_utc"], utc=True, errors="raise")
    print("Période UTC observée :", dates.min(), "→", dates.max())

appels = fenetre.loc[
    fenetre["status"].eq("success") & fenetre["model_version"].eq(VERSION_MODELE)
].iloc[::-1].copy()
entrees = appels["input_json"].map(json.loads).tolist()
if any(not isinstance(entree, dict) or not set(VARIABLES).issubset(entree) for entree in entrees):
    raise ValueError("Un JSON enregistré ne contient pas les trois variables attendues.")

courant = pd.DataFrame(entrees, columns=VARIABLES).astype("float64")
print("Prédictions réussies utilisées pour Evidently, version", VERSION_MODELE, ":", len(courant))
print("Réussites d’autres versions exclues :", int(
    (fenetre["status"].eq("success") & ~fenetre["model_version"].eq(VERSION_MODELE)).sum()
))
if appels.empty:
    print("données insuffisantes : aucune réussite de cette version dans la fenêtre")

Appels /predict dans la fenêtre : 100 / 100
Réussites : 100
Rejets : 0
Pannes : 0
Période UTC observée : 2026-11-20 10:14:45.846000+00:00 → 2026-11-20 10:16:44.936000+00:00
Prédictions réussies utilisées pour Evidently, version 4 : 100
Réussites d’autres versions exclues : 0


## 4. Calculer les scores de référence avec le pipeline existant

Nous avons les entrées de référence, mais il nous manque leurs scores pour comparer les sorties du modèle.

**La cellule suivante** charge une fois l'artefact local utilisé par l'application avec CreditScoringModel,
vérifie sa version et récupère son seuil de décision depuis sa provenance.
Elle relit le train avec **toutes les variables exigées par le pipeline**, dans l'ordre de sa signature,
puis calcule les probabilités de défaut par lots.

Le pipeline réutilise son imputation et son modèle entraînés : **aucun réentraînement**.
Les valeurs originales surveillées à l'étape 2 restent inchangées.
Les contrôles refusent une version différente ou un score absent, infini ou hors de [0, 1].

**À lire :** nombre de scores calculés, version et seuil. Cette cellule peut prendre du temps.
scores_reference décrit les prédictions sur le train, pas les défauts réels de TARGET.
Après remplacement du modèle, cette référence doit être recalculée.

In [4]:
import sys
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
from app.prediction import CreditScoringModel

modele_suivi = CreditScoringModel()
if str(modele_suivi.provenance["model_version"]) != VERSION_MODELE:
    raise ValueError("La version configurée ne correspond pas à l’artefact de l’application.")
seuil_modele = modele_suivi.threshold
scores_lots = []
for lot in pd.read_csv(
    CHEMIN_TRAIN, usecols=modele_suivi.feature_names, chunksize=TAILLE_LOT_PREDICTION
):
    lot = lot.loc[:, modele_suivi.feature_names]
    probabilites = modele_suivi.pipeline.predict_proba(lot)
    scores_lots.append(pd.Series(probabilites[:, modele_suivi.default_class_index]))

scores_reference = pd.concat(scores_lots, ignore_index=True)
if not scores_reference.between(0, 1).all():
    raise ValueError("Le pipeline a produit un score manquant, infini ou hors de [0, 1].")
print("Clients dans la référence des scores :", len(scores_reference))
print("Version :", VERSION_MODELE, "| Seuil :", seuil_modele)

Clients dans la référence des scores : 307510
Version : 4 | Seuil : 0.5


## 5. Construire deux lots comparables et vérifier leurs effectifs

**La cellule suivante** assemble les colonnes nécessaires au rapport :

| Lot | Entrées | Score |
|---|---|---|
| reference_rapport | Les trois variables du train | Calculé à l'étape 4, dans le même ordre de lignes |
| courant_rapport | Les trois variables des réussites API retenues | Enregistré pour ces mêmes appels dans SQLite |

Nous conservons uniquement les réussites de **même version et même seuil** que l'artefact.
Aucune prédiction API n'est recalculée. Nous contrôlons les scores, les classes et leur cohérence :
la classe vaut 1 lorsque le score est supérieur ou égal au seuil.

**À lire :** exclusions pour seuil différent, nombre réellement retenu et tableau effectifs.
Pour comparer une distribution, il faut au moins MIN_OBSERVATIONS valeurs renseignées dans chaque lot.
Une colonne trop peu renseignée est écartée de ce test, mais ses null restent mesurables
si les lots contiennent suffisamment de lignes.

Le minimum de 50 est un garde-fou du PoC, pas une garantie statistique.
En dessous, le notebook indique « données insuffisantes ».

In [5]:
resultats_api = appels.loc[appels["threshold"].eq(seuil_modele)].copy()
print("Réussites exclues pour seuil différent :", len(appels) - len(resultats_api))
print("Réussites retenues pour le rapport unique :", len(resultats_api))
if len(reference) != len(scores_reference):
    raise ValueError("Les entrées et les scores de référence ne sont pas alignés.")

reference_rapport = reference.reset_index(drop=True).copy()
reference_rapport["score"] = scores_reference.to_numpy()
courant_rapport = courant.loc[
    appels["threshold"].eq(seuil_modele).to_numpy()
].reset_index(drop=True).copy()
scores_api = pd.to_numeric(resultats_api["score"], errors="raise").reset_index(drop=True)
classes_api = resultats_api["prediction"].reset_index(drop=True)
if not scores_api.between(0, 1).all() or not classes_api.isin([0, 1]).all():
    raise ValueError("Scores ou classes enregistrés invalides.")
if not classes_api.eq(scores_api.ge(seuil_modele).astype(int)).all():
    raise ValueError("Une classe enregistrée ne correspond pas au score et au seuil.")
courant_rapport["score"] = scores_api

if reference_rapport.isin([float("inf"), float("-inf")]).any().any() or courant_rapport.isin([float("inf"), float("-inf")]).any().any():
    raise ValueError("Valeur infinie détectée avant comparaison.")

effectifs = pd.DataFrame({
    "lignes_reference": len(reference_rapport),
    "valeurs_renseignees_reference": reference_rapport.count(),
    "lignes_api": len(courant_rapport),
    "valeurs_renseignees_api": courant_rapport.count(),
})
effectifs["distribution_comparable"] = (
    effectifs["valeurs_renseignees_reference"].ge(MIN_OBSERVATIONS)
    & effectifs["valeurs_renseignees_api"].ge(MIN_OBSERVATIONS)
)
display(effectifs)
variables_comparables = effectifs.index[effectifs["distribution_comparable"]].tolist()
for variable in effectifs.index:
    if variable not in variables_comparables:
        print(variable, ": données insuffisantes pour comparer les distributions")
lots_suffisants = min(len(reference_rapport), len(courant_rapport)) >= MIN_OBSERVATIONS

Réussites exclues pour seuil différent : 0
Réussites retenues pour le rapport unique : 100


,lignes_reference,valeurs_renseignees_reference,lignes_api,valeurs_renseignees_api,distribution_comparable
AMT_CREDIT,307510,307510,100,100,True
AMT_INCOME_TOTAL,307510,307510,100,100,True
EXT_SOURCE_2,307510,306850,100,100,True
score,307510,307510,100,100,True


## 6. Produire et interpréter le rapport Evidently unique

**La cellule suivante** convertit les deux lots au format Evidently et construit un seul Report.
Elle ajoute le suivi des valeurs manquantes des trois entrées et deux mesures de dérive
pour chaque distribution comparable, y compris celle du score.

| Mesure | Question | Lecture |
|---|---|---|
| MissingValueCount | L'information manque-t-elle plus souvent ? | Count = nombre de null ; Share = proportion entre 0 et 1 |
| Wasserstein normalisée | Quelle est l'ampleur de la différence ? | Distance élevée = distributions plus éloignées ; détection à partir de 0,1 |
| KS | Une différence est-elle statistiquement détectée ? | Petite p-value = différence détectée ; seuil actuel 0,0125 |

**Wasserstein est notre indicateur principal**, KS apporte une information complémentaire.
Le seuil Wasserstein de 0,1 est un point de départ Evidently à calibrer, pas une limite métier validée.
Une distance de 0,2 ne signifie pas « 20 % de clients différents ». Les valeurs extrêmes peuvent peser sur cette distance.

Pour KS, SEUIL_P partage 0,05 entre les quatre distributions prévues : trois entrées et un score.
Cette correction limite les fausses alertes multiples ; elle ne corrige pas les analyses répétées au fil du temps.
Le résultat KS est une p-value, pas une mesure de l'ampleur du changement.

**Comment lire le rapport :**
- Current désigne les appels retenus ; Reference désigne le train.
- Pour les null, comparer surtout Share, car les lots ont des tailles différentes.
  Un affichage arrondi à 0.00 peut masquer une faible proportion non nulle.
- Dans DATA DISTRIBUTION, choisir perc pour comparer des pourcentages plutôt que les effectifs bruts.
- Le graphique DATA DRIFT utilise un index regroupé : son axe horizontal n'est pas une date.
- Les deux méthodes affichent les mêmes distributions, avec des résultats de détection différents.

Les tests automatiques sur les null sont désactivés en attendant un seuil sur leurs proportions.
Evidently signale séparément les détections KS et Wasserstein ; aucune alerte globale combinée n'est définie.
Un signal concordant mérite une investigation, mais ne démontre pas une baisse des performances du modèle.

In [ ]:
rapport_drift = None
if not lots_suffisants:
    print("données insuffisantes pour le rapport Evidently")
else:
    definition_rapport = DataDefinition(numerical_columns=VARIABLES + ["score"])
    reference_evidently = Dataset.from_pandas(reference_rapport, data_definition=definition_rapport)
    courant_evidently = Dataset.from_pandas(courant_rapport, data_definition=definition_rapport)
    metriques = [
        MissingValueCount(column=variable, tests=[], share_tests=[])
        for variable in VARIABLES
    ]
    for variable in variables_comparables:
        metriques.extend([
            ValueDrift(column=variable, method="wasserstein", threshold=SEUIL_WASSERSTEIN),
            ValueDrift(column=variable, method="ks", threshold=SEUIL_P),
        ])
    print("Wasserstein (principal) : distance >= ", SEUIL_WASSERSTEIN)
    print("KS (complémentaire) : p-value < ", SEUIL_P)
    rapport_drift = Report(metriques, include_tests=True).run(
        current_data=courant_evidently, reference_data=reference_evidently
    )
    display(rapport_drift)

## 7. Lire la proportion de clients classés « défaut prédit »

**Pourquoi ce tableau en plus du rapport ?** Il traduit les scores en une décision facile à expliquer :
quelle part des clients dépasse le seuil de défaut ?

**La cellule suivante** utilise exactement les mêmes lots que le rapport et affiche leurs volumes,
le score moyen et la proportion de classes 1. Elle calcule ensuite :
**proportion API moins proportion de référence**, en points de pourcentage.

Par exemple, passer de 15 % à 25 % représente **+10 points**, pas +10 %.
Une hausse signifie que le modèle classe plus souvent les clients reçus en défaut prédit.
Il faut examiner leurs profils et leurs données ; ce ne sont pas des défauts réellement constatés.
Ce suivi reste descriptif, sans seuil automatique. Avec trop peu de lignes : « données insuffisantes ».

In [7]:
if not lots_suffisants:
    print("données insuffisantes pour comparer les classes prédites")
else:
    proportion_reference = scores_reference.ge(seuil_modele).mean()
    proportion_api = classes_api.eq(1).mean()
    display(pd.DataFrame({
        "lot": ["Train (référence)", "Appels API"],
        "nombre_clients": [len(scores_reference), len(scores_api)],
        "score_moyen": [scores_reference.mean(), scores_api.mean()],
        "defauts_predits_pct": [100 * proportion_reference, 100 * proportion_api],
    }).set_index("lot"))
    print("Écart API − référence :", round(100 * (proportion_api - proportion_reference), 2),
          "points de pourcentage")

,nombre_clients,score_moyen,defauts_predits_pct
lot,,,
Train (référence),307510,0.380730,30.425352
Appels API,100,0.379184,32.000000


Écart API − référence : 1.57 points de pourcentage


## 8. Comparer le fonctionnement du service

Le train ne contient pas d'erreurs API ni de durées : **la référence est ici la fenêtre précédente**.
Nous utilisons tous les statuts et toutes les versions des deux fenêtres lues à l'étape 3,
pas seulement les réussites retenues pour Evidently.

**La cellule suivante** définit une fonction commune aux deux lots, contrôle statuts et durées,
puis affiche les périodes, les versions, les effectifs et les indicateurs :

| Indicateur | Calcul | Unité de l'écart actuel moins précédent |
|---|---|---|
| Taux de rejet | rejected / total des appels enregistrés | Points de pourcentage |
| Taux de panne | error / total des appels enregistrés | Points de pourcentage |
| Durée médiane | 50e percentile des durées des réussites | Millisecondes |
| Durée P95 | 95e percentile des durées des réussites | Millisecondes |

La médiane décrit la durée centrale ; le P95 renseigne sur les traitements les plus lents.
Les percentiles sont calculés par interpolation avec pandas.
Chaque valeur exige MIN_OBSERVATIONS appels ou réussites selon l'indicateur.
Les écarts exigent aussi **deux fenêtres complètes**, soit 200 appels avec notre réglage.
Sinon : « données insuffisantes ». Le P95 d'un petit lot reste fluctuant.

**À lire :** un écart positif indique une hausse, pas automatiquement une anomalie.
Une hausse des rejets invite à examiner les motifs de validation ; une hausse des pannes, les erreurs serveur.
Une hausse des durées invite à vérifier charge, concurrence, ressources et changements de version.
La fenêtre précédente n'est pas nécessairement saine : aucun seuil opérationnel fixe n'est encore défini.

**Périmètre de la mesure :** duration_ms couvre le traitement côté application avant écriture SQLite,
sans attente dans la file Gradio ni trajet réseau.
Les appels non parvenus au traitement, certains JSON rejetés en amont et les échecs de stockage
peuvent être absents de SQLite. Ces taux ne mesurent donc pas la disponibilité complète du service.

In [8]:
def indicateurs_service(lot):
    """Calcule les mesures et leurs effectifs pour une fenêtre enregistrée."""
    if not lot["status"].isin(["success", "rejected", "error"]).all():
        raise ValueError("Statut inconnu dans une fenêtre.")
    durees = pd.to_numeric(
        lot.loc[lot["status"].eq("success"), "duration_ms"], errors="raise"
    )
    if durees.isna().any() or durees.isin([float("inf"), float("-inf")]).any() or durees.lt(0).any():
        raise ValueError("Une durée de réussite est absente ou invalide.")
    total = len(lot)
    mesures = {}
    for statut, nom in [("rejected", "Taux de rejet"), ("error", "Taux de panne")]:
        valeur = 100 * lot["status"].eq(statut).sum() / total if total >= MIN_OBSERVATIONS else None
        mesures[nom] = (valeur, total, "%", "points de pourcentage")
    for quantile, nom in [(0.5, "Durée médiane"), (0.95, "Durée P95")]:
        valeur = durees.quantile(quantile) if len(durees) >= MIN_OBSERVATIONS else None
        mesures[nom] = (valeur, len(durees), "ms", "ms")
    return mesures


for nom, lot in [("Précédente", fenetre_precedente), ("Actuelle", fenetre)]:
    print(nom, ":", len(lot), "/", NOMBRE_APPELS, "appels")
    if not lot.empty:
        dates_lot = pd.to_datetime(lot["timestamp_utc"], utc=True, errors="raise")
        print("  Période UTC :", dates_lot.min(), "→", dates_lot.max())
        print("  Versions des réussites :",
              lot.loc[lot["status"].eq("success"), "model_version"].value_counts().to_dict())

precedent = indicateurs_service(fenetre_precedente)
actuel = indicateurs_service(fenetre)
fenetres_completes = len(fenetre) == len(fenetre_precedente) == NOMBRE_APPELS
if not fenetres_completes:
    print("données insuffisantes : deux fenêtres complètes sont nécessaires pour les écarts.")

lignes_service = []
for nom, (valeur_actuelle, effectif_actuel, unite, unite_ecart) in actuel.items():
    valeur_precedente, effectif_precedent, _, _ = precedent[nom]
    comparable = fenetres_completes and valeur_actuelle is not None and valeur_precedente is not None
    lignes_service.append({
        "indicateur": nom,
        "precedent": f"{valeur_precedente:.2f} {unite}" if valeur_precedente is not None else "données insuffisantes",
        "effectif_precedent": effectif_precedent,
        "actuel": f"{valeur_actuelle:.2f} {unite}" if valeur_actuelle is not None else "données insuffisantes",
        "effectif_actuel": effectif_actuel,
        "ecart": f"{valeur_actuelle - valeur_precedente:+.2f} {unite_ecart}" if comparable else "données insuffisantes",
    })
display(pd.DataFrame(lignes_service).set_index("indicateur"))
print("Une hausse est un signal à examiner ; aucun seuil automatique n'est encore défini.")

Précédente : 100 / 100 appels
  Période UTC : 2026-11-20 07:48:29.965000+00:00 → 2026-11-20 07:50:24.865000+00:00
  Versions des réussites : {'4': 100}
Actuelle : 100 / 100 appels
  Période UTC : 2026-11-20 10:14:45.846000+00:00 → 2026-11-20 10:16:44.936000+00:00
  Versions des réussites : {'4': 100}


,precedent,effectif_precedent,actuel,effectif_actuel,ecart
indicateur,,,,,
Taux de rejet,0.00 %,100,0.00 %,100,+0.00 points de pourcentage
Taux de panne,0.00 %,100,0.00 %,100,+0.00 points de pourcentage
Durée médiane,39.07 ms,100,43.30 ms,100,+4.23 ms
Durée P95,54.75 ms,100,49.20 ms,100,-5.55 ms


Une hausse est un signal à examiner ; aucun seuil automatique n'est encore défini.


## 9. Préparer la synthèse et connaître les limites

À partir des sorties précédentes, répondre dans cet ordre :

1. **Périmètre** : combien d'appels, quelle période observée, quelle version et quel seuil ?
   Combien de réussites sont réellement comparables ?
2. **Données** : quelles variables changent selon Wasserstein et KS ?
   Les graphiques montrent-ils un déplacement des montants ou une autre répartition ?
   Les proportions de null augmentent-elles ?
3. **Décisions** : les scores et la proportion de classes 1 évoluent-ils ?
4. **Service** : rejets, pannes ou durées augmentent-ils par rapport aux appels précédents ?
5. **Investigation** : pour chaque signal, quelle donnée, erreur ou condition de fonctionnement faut-il examiner ?

**Limites à présenter à l'oral :**
- Le train préparé est notre référence choisie, pas une preuve de fonctionnement normal en production.
  Ses colonnes ne suffisent pas à prouver qu'il correspond exactement au sous-ensemble utilisé pour entraîner la v4.
- Le simulateur utilise par défaut le CSV test ; SQLite ne conserve pas son nom.
  Des clients peuvent être répétés. Tirer davantage de clients du même fichier ne crée pas forcément une dérive temporelle.
- Les petits lots fluctuent et deux exécutions successives peuvent partager des appels.
  Les seuils sont exploratoires. Sans défauts réels observés, nous ne mesurons ni précision ni dégradation du modèle.

Ce notebook produit une analyse lors de son exécution : ce n'est pas encore un script planifié,
une alerte automatique complète ou un export HTML enregistré sur disque.

Documentation utilisée pour Evidently **0.7.23** :
- [Version et code source](https://github.com/evidentlyai/evidently/tree/v0.7.23)
- [Métriques de qualité des données](https://docs.evidentlyai.com/metrics/all_metrics#column-data-quality)
- [Méthodes et seuils de dérive](https://docs.evidentlyai.com/metrics/customize_data_drift)